# 09.3 翻唱识别：同一首作品的不同演绎

上一 Notebook 的峰值配对指纹要求查询与库内条目来自同一源录音，因此不适合匹配重新演唱或演奏的版本。翻唱识别（cover song identification）关注作品层面的关系：同一作品的不同演绎可能同时改变音色、调性、速度、结构细节和伴奏编配。

本 Notebook 用色度（chroma）近似表示音级随时间的分布。色度折叠八度，可减弱部分八度和音色差异，但谱包络、调律、演奏法与混音仍会改变各维能量，因此它不是严格的音色不变量。

流程分四步：计算节拍同步色度；用两首歌的全局色度向量估计移调；把估计移位施加到查询的节拍序列；用子序列 DTW 对齐。后续实验逐项检查这些步骤在受控版本上的作用和限制。


## 0. 环境与合成翻唱库

翻唱往往同时改变多个因素，难以构成严格的单变量对照，完整录音的再分发也可能受版权和许可限制。因此，本章先用合成版本检查各处理步骤在受控条件下的行为。

第三章的渲染程序为八首旋律各生成六个版本：原版之外，分别改变音色、调性和速度，再加入低八度叠置与重配和弦两个版本，共 48 条音频。

根据变换定义，可预期调性变化主要平移音级位置，速度变化主要拉伸时间轴，低八度叠置经音级折叠后较接近原版；音色仍可能改变色度配比和拍点估计，重配和弦会直接改变和声音级。实验用于检验这些预期。

变速版本使用非整数倍速度比，并加入轻微节奏弹性，以同时检查节拍同步与 DTW，而不只测试理想的恒定时间缩放。


In [ ]:
from pathlib import Path
import sys
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

# 路径推断：从 cwd 向上找含 CODE/chapter09/_common 的目录；ROOT 指向 CODE/chapter09/
_p = Path.cwd()
while not (_p / "CODE" / "chapter09" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter09/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter09"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from _common.audio_io import load_audio  # audio_io 先设 NUMBA_CACHE_DIR,须早于 librosa 导入
from _common.cover_render import VERSION_PRESETS, CoverSpec, render_cover, render_cover_set
from _common.env_check import check_notebook_env
from _common.paths import portable_path
from _common.plotting import GRAY_IMAGE_CMAP, LINE_GRAYS, finish_figure, setup_plot_style

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 民歌 MIDI 的 tempo 事件不在第 0 轨，pretty_midi 会告警，读取不受影响
warnings.filterwarnings("ignore", message=r"Tempo, Key or Time signature.*", category=RuntimeWarning)
check_notebook_env("09_3_cover_song")

SR = 22050
HOP = 512
FRAME_RATE = SR / HOP

DATASETS = ROOT.parent / "datasets"
MELODY_DIR = DATASETS / "melodies"
AUTHOR_COVER_DIR = DATASETS / "audio_author" / "chapter_09_author" / "cover"
COVER_DIR = ROOT / "output_audio" / "09_3"
OUTPUT_FIGURES = ROOT / "output_figures"
OUTPUT_TABLES = ROOT / "outputs" / "tables"
for path in [COVER_DIR, OUTPUT_FIGURES, OUTPUT_TABLES]:
    path.mkdir(parents=True, exist_ok=True)
setup_plot_style()

def rel(path):
    return portable_path(path, ROOT)

# 八首旋律 × 六版本并行渲染（FluidSynth 子进程，线程池即可）
midi_paths = sorted(MELODY_DIR.glob("*.midi"))
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(render_cover_set, p, COVER_DIR, None, SR) for p in midi_paths]
    cover_records = [rec for fut in futures for rec in fut.result()]
cover_df = pd.DataFrame(cover_records)
print(f"渲染 {len(cover_df)} 个版本({len(midi_paths)} 首 × {len(VERSION_PRESETS)} 版)，耗时 {time.perf_counter() - t0:.1f} 秒")
print(cover_df.groupby("version").size().to_string())


## 1. 色度与节拍同步

色度把每个时刻的频谱能量按十二个音级折叠：同一音级在不同八度的能量汇入同一维。不同乐器的谐波结构仍会改变各维比例，因此色度主要减弱八度差异，并只能部分减弱音色差异。

谐波音级轮廓（harmonic pitch class profile, HPCP）是一种常用色度表示，它根据谱峰的位置与权重向音级投射。本 Notebook 使用 `librosa` 的 CQT 色度，目标相近，但具体计算方式不同。

固定帧移下，色度序列长度随录音时长变化。节拍同步在每个估计节拍区间内对色度帧取中位数，得到每拍一个向量。若两个版本的拍点都估计合理，全局速度差异会在较大程度上减弱。

节拍同步依赖拍点估计。拍点误差会降低两条序列的一致性，后续 DTW 只能补偿部分局部错位和节奏弹性。下方原版与移调版的标称速度相同，估计 BPM 和节拍数却不同，也说明这一步不是确定性的速度归一化。


In [ ]:
def beat_sync_chroma(y, sr=SR):
    # CQT 色度 → 节拍区间中位数聚合，返回每拍一帧的 12 维序列与估计速度
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=HOP)
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP)
    if len(beats) < 4:
        # 回退返回帧级色度，下游 OTI/DTW 的量纲随之变化；实践中渲染音频不会触发
        return chroma, float(np.atleast_1d(tempo)[0])
    return librosa.util.sync(chroma, beats, aggregate=np.median), float(np.atleast_1d(tempo)[0])


# 《茉莉花》原版与移调版对照：旋律与和声配置相同，整体上移四个半音
y_v0, _ = load_audio(COVER_DIR / "茉莉花" / "v0_original.wav", sr=SR)
y_v2, _ = load_audio(COVER_DIR / "茉莉花" / "v2_key.wav", sr=SR)
chroma_v0, tempo_v0 = beat_sync_chroma(y_v0)
chroma_v2, tempo_v2 = beat_sync_chroma(y_v2)
print(f"v0 估计速度 {tempo_v0:.0f} BPM，{chroma_v0.shape[1]} 拍；v2 {tempo_v2:.0f} BPM，{chroma_v2.shape[1]} 拍")

pitch_classes = ["C", "C♯", "D", "D♯", "E", "F", "F♯", "G", "G♯", "A", "A♯", "B"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, mat, title in [(axes[0], chroma_v0, "v0 原版"), (axes[1], chroma_v2, "v2 移高四个半音")]:
    ax.imshow(mat, origin="lower", aspect="auto", cmap=GRAY_IMAGE_CMAP)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("拍")
    ax.set_yticks(range(12), pitch_classes)
axes[0].set_ylabel("音级")
finish_figure(fig, OUTPUT_FIGURES / "09_3_chroma_transpose.png")
plt.show()


## 2. OTI 与移调归一

整体移调在色度上近似表现为沿循环音级轴平移，因此可以比较十二种候选移位。这里参考 Serrà 等人的最优移调指数（optimal transposition index, OTI），在两首歌各自的全局色度向量之间估计移位量。

全局色度向量由节拍同步色度沿时间轴取平均，再做 L1 归一化得到。对其中一个向量作十二次循环移位，取与另一个点积最大的移位数：

OTI(g_A, g_B) = argmax_i ⟨ circshift(g_A, i), g_B ⟩

OTI 在时间平均后的单个向量上估计，此时尚未比较完整序列。这里用 L1 归一向量的点积；它不等同于余弦相似度，只有 L2 归一向量的点积才与余弦等价。

估计出移位后，再把它施加到查询的节拍同步色度序列。移位正方向是实现约定；若约定相反，指数会变为模 12 意义下的相反数，只要估计与应用使用同一约定，归一结果不受影响。

OTI 不需要训练，但仍依赖色度表示、全局聚合和点积准则。若全局音级分布含混，或两个版本的和声差异很大，移位也可能估错。


In [ ]:
def global_chroma(beat_chroma):
    # 时间平均 + L1 归一，压成单个 12 维向量
    g = beat_chroma.mean(axis=1)
    return g / g.sum()


def optimal_transposition_index(g_a, g_b):
    # 十二次循环移位，取与 g_b 点积最大者（Serrà–Gómez–Herrera 原始定义）
    dots = [float(np.dot(np.roll(g_a, i), g_b)) for i in range(12)]
    return int(np.argmax(dots)), dots


g_v0 = global_chroma(chroma_v0)
g_v2 = global_chroma(chroma_v2)
oti, dots = optimal_transposition_index(g_v2, g_v0)
print("v2 对 v0 的逐移位点积:")
print(pd.DataFrame({"移位": range(12), "点积": [round(d, 4) for d in dots]}).to_string(index=False))
print(f"OTI = {oti}(循环移位 {oti} 步 ≡ 反向 {(-oti) % 12} 个半音,与渲染移高 4 个半音互逆)")


## 3. 对齐与检索

OTI 移位后，需要对齐两条十二维节拍序列。这里沿用子序列 DTW，把局部代价从一维半音差改为余弦距离，即非零向量上的 1 减去余弦相似度。

检索时，查询与库内每首原版分别估计 OTI，再计算子序列 DTW 的每步平均代价，并按代价从小到大排序。

48 条合成查询中，原版、音色变化、移调、变速和低八度叠置五组均为 8/8 命中，重配和弦组为 4/8。结果的适用范围限于这些受控音频，不支持对真实翻唱或更大扰动范围作同样判断。


In [ ]:
def cover_pipeline_cost(query_chroma, lib_chroma):
    # OTI 归一 + 子序列 DTW（局部代价 1-cos），返回每步平均代价
    oti_q, _ = optimal_transposition_index(global_chroma(query_chroma), global_chroma(lib_chroma))
    shifted = np.roll(query_chroma, oti_q, axis=0)
    D, wp = librosa.sequence.dtw(X=shifted, Y=lib_chroma, metric="cosine", subseq=True)
    return float(D[-1].min()) / len(wp)


# 库：八首 v0 原版；查询：全部 48 个版本
lib_chromas, lib_tempos = {}, {}
for song in cover_df["song"].unique():
    y_lib, _ = load_audio(COVER_DIR / song / "v0_original.wav", sr=SR)
    lib_chromas[song], lib_tempos[song] = beat_sync_chroma(y_lib)

axis_of = {"v0_original": "原版", "v1_timbre": "音色", "v2_key": "调性", "v3_tempo": "速度",
           "v4_texture_octave": "织体(叠八度)", "v5_texture_reharm": "织体(换和声)"}
retrieval_rows = []
t0 = time.perf_counter()
for _, row in cover_df.iterrows():
    y_q, _ = load_audio(row["wav"], sr=SR)
    chroma_q, _ = beat_sync_chroma(y_q)
    costs = {song: cover_pipeline_cost(chroma_q, lib_c) for song, lib_c in lib_chromas.items()}
    best = min(costs, key=costs.get)
    retrieval_rows.append(
        {"song": row["song"], "version": row["version"], "轴": axis_of[row["version"]],
         "best": best, "命中": best == row["song"], "best_cost": round(costs[best], 4),
         "truth_cost": round(costs[row["song"]], 4)}
    )
print(f"48 条查询检索耗时 {time.perf_counter() - t0:.1f} 秒")
retrieval_df = pd.DataFrame(retrieval_rows)
axis_summary = retrieval_df.groupby("轴")["命中"].agg(["sum", "count"])
axis_summary["准确率"] = (axis_summary["sum"] / axis_summary["count"]).round(3)
print(axis_summary[["sum", "count", "准确率"]].to_string())
print()
print("未命中明细:")
print(retrieval_df[~retrieval_df["命中"]][["song", "version", "best", "best_cost", "truth_cost"]].to_string(index=False))


## 4. 移调扫描与 OTI 消融

检索实验中的八个移调版本全部命中，但原因可能包括 OTI，也可能包括八首候选本来就容易区分。为单独观察 OTI，本节把《茉莉花》渲染为 0–11 半音的十二个版本，并保持音色、速度、伴奏与和声配置不变。

每个版本与原版计算两种代价：包含 OTI 的完整流程，以及省略 OTI 后直接对齐的 DTW。除零移位外，带 OTI 的代价为 0.027–0.079，未使用 OTI 时为 0.314–0.889。

在这项单曲消融中，OTI 明显降低了所有非零移调版本的代价。这个结果支持 OTI 是当前流程处理整体移调的主要步骤，但无法说明未归一版本在任何候选库中都一定检索失败。


In [ ]:
# 十二个移调版本（音色、速度同原版，只动调性）
oti_specs = [CoverSpec(version=f"t{t:02d}", transpose=t, seed=500 + t) for t in range(12)]
with ThreadPoolExecutor(max_workers=8) as pool:
    oti_wavs = list(
        pool.map(
            lambda spec: render_cover(MELODY_DIR / "茉莉花.midi", spec, COVER_DIR / "茉莉花_oti" / f"{spec.version}.wav", sr=SR),
            oti_specs,
        )
    )

chroma_lib_moli = lib_chromas["茉莉花"]
oti_rows = []
for spec, rec in zip(oti_specs, oti_wavs):
    y_t, _ = load_audio(rec["wav"], sr=SR)
    chroma_t, _ = beat_sync_chroma(y_t)
    cost_oti = cover_pipeline_cost(chroma_t, chroma_lib_moli)
    D, wp = librosa.sequence.dtw(X=chroma_t, Y=chroma_lib_moli, metric="cosine", subseq=True)
    cost_raw = float(D[-1].min()) / len(wp)
    oti_rows.append({"移调（半音）": spec.transpose, "OTI 归一": round(cost_oti, 4), "未归一": round(cost_raw, 4)})
oti_df = pd.DataFrame(oti_rows)
print(oti_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(oti_df["移调（半音）"], oti_df["OTI 归一"], marker="o", color=LINE_GRAYS[0], label="OTI 归一")
ax.plot(oti_df["移调（半音）"], oti_df["未归一"], marker="s", color=LINE_GRAYS[3], label="未归一")
ax.set_xlabel("移调量（半音）")
ax.set_ylabel("每步平均代价")
ax.set_xticks(range(12))
ax.legend(fontsize=8)
finish_figure(fig, OUTPUT_FIGURES / "09_3_oti_scan.png")
plt.show()


## 5. 速度扫描与节拍同步消融

速度变化先由节拍同步减弱，再由子序列 DTW 处理剩余错位。左图比较两种不使用 DTW 的诊断量：帧级色度按同索引计算的平均余弦，以及节拍同步后按同索引计算的平均余弦。两者都把序列截到较短者。

1.37 倍速版本的估计速度为 161.5 BPM，接近标称 164 BPM；节拍同步后的同索引余弦为 0.751。1.10 与 1.60 倍速版本也估为 161.5 BPM，与标称 132 和 192 BPM 不一致；同步后的余弦分别为 0.374 和 0.410。

仅凭全局 BPM 无法判断拍点误差来源，节拍层级歧义和局部偏移都可能参与。完整流程在四个速度比下的 DTW 代价为 0~0.068，说明它能处理这组合成版本中节拍同步后留下的错位，但单曲实验不足以证明对任意拍点错误都可靠。


In [ ]:
# 四个速度档：1.0(原版)、1.1、1.37、1.6 倍，带轻微节奏弹性
tempo_specs = [
    CoverSpec(version="v0_original", seed=610),
    CoverSpec(version="x110", tempo_ratio=1.1, rubato=0.03, seed=611),
    CoverSpec(version="v3_tempo", tempo_ratio=1.37, rubato=0.03, seed=612),
    CoverSpec(version="x160", tempo_ratio=1.6, rubato=0.03, seed=613),
]
with ThreadPoolExecutor(max_workers=8) as pool:
    tempo_wavs = list(
        pool.map(
            lambda spec: render_cover(MELODY_DIR / "茉莉花.midi", spec, COVER_DIR / "茉莉花_tempo" / f"{spec.version}.wav", sr=SR),
            tempo_specs,
        )
    )

def frame_cosine(chroma_a, chroma_b):
    # 逐帧一一对应求余弦，截到短者；等长假设的最朴素基线
    n = min(chroma_a.shape[1], chroma_b.shape[1])
    a, b = chroma_a[:, :n], chroma_b[:, :n]
    num = (a * b).sum(axis=0)
    den = np.linalg.norm(a, axis=0) * np.linalg.norm(b, axis=0) + 1e-12
    return float((num / den).mean())


chroma_lib_frames = librosa.feature.chroma_cqt(y=load_audio(COVER_DIR / "茉莉花" / "v0_original.wav", sr=SR)[0], sr=SR, hop_length=HOP)
tempo_rows = []
for spec, rec in zip(tempo_specs, tempo_wavs):
    y_t, _ = load_audio(rec["wav"], sr=SR)
    chroma_t_frames = librosa.feature.chroma_cqt(y=y_t, sr=SR, hop_length=HOP)
    chroma_t_beats, tempo_est = beat_sync_chroma(y_t)
    tempo_rows.append(
        {
            "速度比": spec.tempo_ratio,
            "标称BPM": round(120 * spec.tempo_ratio),
            "估计BPM": round(tempo_est, 1),
            "帧级逐帧余弦": round(frame_cosine(chroma_t_frames, chroma_lib_frames), 3),
            "节拍同步逐帧余弦": round(frame_cosine(chroma_t_beats, chroma_lib_moli), 3),
            "完整管线代价": round(cover_pipeline_cost(chroma_t_beats, chroma_lib_moli), 4),
        }
    )
tempo_df = pd.DataFrame(tempo_rows)
print(tempo_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(tempo_df["速度比"], tempo_df["帧级逐帧余弦"], marker="s", color=LINE_GRAYS[3], label="帧级同索引")
axes[0].plot(tempo_df["速度比"], tempo_df["节拍同步逐帧余弦"], marker="o", color=LINE_GRAYS[0], label="节拍同步后同索引")
axes[0].set_xlabel("速度比")
axes[0].set_ylabel("同索引余弦相似度")
axes[0].set_ylim(0, 1.02)
axes[0].legend(fontsize=8)
axes[1].plot(tempo_df["速度比"], tempo_df["完整管线代价"], marker="o", color=LINE_GRAYS[0])
axes[1].set_xlabel("速度比")
axes[1].set_ylabel("完整管线 DTW 代价")
finish_figure(fig, OUTPUT_FIGURES / "09_3_tempo_ablation.png")
plt.show()


## 6. 伴奏与和声变化的逐项对照

本节为《茉莉花》构造五个版本：无伴奏、旋律低八度叠置、主音与属音持续低音、按旋律配置三和弦，以及换用平行小调重配和弦。
低八度叠置几乎不改变音级类别；持续低音提高主音与属音的相对能量；三和弦和重配和弦以不同方式加入或改变和声音级。
当前结果中，持续低音版本对原版的代价为 0.203，第一名变为《友谊地久天长》；三和弦版本代价为 0.173，仍返回《茉莉花》；重配和弦版本代价为 0.232，返回《红沙拉帆》。

In [ ]:
# 五个伴奏与和声版本（原版与两端的版本复用前面渲染，中间两项补渲）
texture_extra = [CoverSpec(version="drone", texture="drone", seed=701), CoverSpec(version="triads", texture="triads", seed=702)]
with ThreadPoolExecutor(max_workers=8) as pool:
    list(
        pool.map(
            lambda spec: render_cover(MELODY_DIR / "茉莉花.midi", spec, COVER_DIR / "茉莉花_texture" / f"{spec.version}.wav", sr=SR),
            texture_extra,
        )
    )
texture_wavs = [
    ("无伴奏", COVER_DIR / "茉莉花" / "v0_original.wav"),
    ("低八度叠置", COVER_DIR / "茉莉花" / "v4_texture_octave.wav"),
    ("持续低音", COVER_DIR / "茉莉花_texture" / "drone.wav"),
    ("配三和弦", COVER_DIR / "茉莉花_texture" / "triads.wav"),
    ("重配和弦", COVER_DIR / "茉莉花" / "v5_texture_reharm.wav"),
]
texture_rows = []
for label, wav in texture_wavs:
    y_t, _ = load_audio(wav, sr=SR)
    chroma_t, _ = beat_sync_chroma(y_t)
    costs = {song: cover_pipeline_cost(chroma_t, lib_c) for song, lib_c in lib_chromas.items()}
    best = min(costs, key=costs.get)
    texture_rows.append(
        {"织体": label, "对原版代价": round(costs["茉莉花"], 4), "检索判定": best, "仍为茉莉花": best == "茉莉花"}
    )
texture_df = pd.DataFrame(texture_rows)
print(texture_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.scatter(range(len(texture_df)), texture_df["对原版代价"], s=48, color=LINE_GRAYS[0])
ax.set_xticks(range(len(texture_df)), texture_df["织体"])
ax.set_ylabel("对原版的每步平均代价")
finish_figure(fig, OUTPUT_FIGURES / "09_3_texture_ablation.png")
plt.show()


## 7. 真实录音、基准与表征学习

合成版本便于控制变量，但无法充分复现自然演奏、录音环境、混音和乐器组合共同造成的变化。把翻唱录音放入 `CODE/datasets/audio_author/chapter_09_author/cover` 后，可运行下方对照单元。一两条录音只适合检查流程，不足以验证真实翻唱性能。

covers80 含 80 首作品的两种演绎，共 160 个 32 kbps、16 kHz 单声道录音；官方页面未明示统一开放许可。SHS-100K 2025 版提供约一万件作品和十万级表演的元数据及 YouTube ID，不分发音频；元数据采用 CC BY-NC 4.0。

Da-TACOS 分发预抽特征与元数据，不含音频。其基准子集含一万五千首，分析子集含一万首，特征包括 HPCP、MFCC 和节奏类特征；元数据与预抽特征采用 CC BY-NC-SA 4.0。使用这些数据前，仍需分别核对数据许可、媒体权利和平台条款。

手工流程之外，也可用表征学习构造检索特征。CQT-Net 从常数 Q 变换谱图学习嵌入，CoverHunter 加入注意力与对齐模块。比较两类方法时，应尽量统一训练数据、目标基准、评价协议和计算条件，并在目标音乐与目标扰动上验证。


In [ ]:
# 真实录音对照：翻唱入库后重跑本单元
author_covers = sorted(AUTHOR_COVER_DIR.glob("*.wav")) if AUTHOR_COVER_DIR.exists() else []
if not author_covers:
    print(f"翻唱尚未入库(期待位置 {rel(AUTHOR_COVER_DIR)}),本单元仅报告状态")
for wav_path in author_covers:
    y_real, _ = load_audio(wav_path, sr=SR)
    chroma_real, _ = beat_sync_chroma(y_real)
    costs = sorted(
        ((song, cover_pipeline_cost(chroma_real, lib_c)) for song, lib_c in lib_chromas.items()),
        key=lambda r: r[1],
    )
    print(f"{wav_path.stem} 的 Top-3: {[(s, round(c, 4)) for s, c in costs[:3]]}")

retrieval_df.to_csv(OUTPUT_TABLES / "09_3_retrieval_by_axis.csv", index=False)
oti_df.to_csv(OUTPUT_TABLES / "09_3_oti_scan.csv", index=False)
tempo_df.to_csv(OUTPUT_TABLES / "09_3_tempo_ablation.csv", index=False)
texture_df.to_csv(OUTPUT_TABLES / "09_3_texture_ablation.csv", index=False)
print("表格已写入", rel(OUTPUT_TABLES))
print("图已写入", rel(OUTPUT_FIGURES))
